# Benchmark 2×6 (Raw vs Curated) — seed 85 — Kaggle T4 ×2 + Camber Stash

Lặp lại benchmark 2×6 (raw/curated × 6 kiến trúc = **12 run**) với **`seed=85`** để đo độ dao động giữa các seed so với bản seed 42 gốc
(`benchmakring_transmission_line_defect_model/benchmark_pipeline.ipynb` cho YOLO, `kaggle_train_rtdetr_peking.ipynb` cho RT-DETR).

Siêu tham số giữ nguyên bản seed 42:

| Nhóm | Thiết lập |
|---|---|
| YOLO11n/m, YOLO26n/m | ultralytics 8.4.172, 50 epoch, 640 px, AdamW lr0=0.001, lrf=0.01, cos_lr, warmup 3, wd 0.0005, close_mosaic 10, AMP, `deterministic=True`, batch 16 (n) / 8 (m) |
| RT-DETR-R50vd / R101vd | PekingU (HF transformers <4.48), 50 epoch, 640 px, AdamW (backbone 1e-4, head 1e-3, wd 0.0005), cosine (eta_min 1e-5), AMP, clip 0.1, **1 run trên 2 GPU**: 2 GPU × batch 8 × accum 1 (R50) / 2 GPU × batch 4 × accum 2 (R101) = effective batch 16, best theo val loss |

**Sửa ổn định RT-DETR (cập nhật 16/11/2026):** giữ BatchNorm ở chế độ train theo RT-DETR gốc (freeze running stats từng làm R50 học kém); trước forward AMP có snapshot các buffer BN và khôi phục chúng nếu loss/box không finite rồi mới retry batch đó bằng fp32. Pipeline vẫn có warmup tuyến tính 2000 bước optimizer (`RTDETR_WARMUP`), kiểm tra loss/gradient finite, gradient clipping và tùy chọn `RTDETR_AMP=0`. Chỉ đặt `RTDETR_FREEZE_BN=1` để chẩn đoán, không dùng cho run benchmark.

**Sửa YOLO done giả (10/11/2026):** 4 run YOLO-m bị ghi `done.json` khi mới 31–42 epoch. Nguyên nhân khả năng cao: checkpoint tạo ở đường dẫn cũ, Ultralytics resume theo `project/save_dir` lưu trong checkpoint nên train tiếp sang thư mục cũ, còn script đọc `results.csv` ở thư mục mới rồi ghi done. Đã sửa: (1) ghi lại đường dẫn trong `last.pt` trước khi resume, (2) dừng nếu Ultralytics ghi sang thư mục khác, (3) chỉ ghi `done.json` khi đủ EPOCHS dòng, (4) `is_done()` kiểm tra số epoch và tự đổi done giả thành `done_invalid.json`, (5) tắt callback Ray Tune.

**RT-DETR chạy 1 model trên 2 GPU (12/11/2026):** mỗi GPU một tiến trình (torchrun), mỗi GPU xử lý một nửa mỗi micro-batch, gradient được cộng gộp giữa 2 GPU ở mỗi bước optimizer rồi cả hai cập nhật giống hệt nhau. Mỗi bước forward gồm 8 ảnh (R50) / 4 ảnh (R101) như bản seed 42 thay vì 2 / 1 ảnh như trước. Batch nào NaN ở fp16 được chạy lại fp32 ngay trên GPU đó; nếu vẫn NaN thì chỉ batch đó bị bỏ. Hết VRAM thì hàng đợi tự chia đôi batch/GPU và chạy lại. Không còn khôi phục RNG state từ checkpoint (RNG đặt lại theo seed/epoch/GPU ở đầu mỗi epoch).

**Tăng tốc RT-DETR theo seed 42 (14/11/2026), vẫn 1 run trên 2 GPU:** seed 36 đo khoảng 40 phút/epoch R50 trên 2 GPU, gần như không nhanh hơn 1 GPU vì nghẽn ở nạp ảnh. Đã đổi theo `kaggle_train_rtdetr_peking.ipynb` (seed 42): (1) chép dataset của run RT-DETR sang `/tmp` (NVMe) trước khi train thay vì đọc `/kaggle/input`, (2) micro-batch mỗi GPU bằng seed 42: R50 8 (accum 1), R101 4 (accum 2), effective batch vẫn 16, (3) bỏ `cudnn.deterministic=True` (seed 42 dùng mặc định), (4) DataLoader giữ worker giữa các epoch. Các sửa ổn định NaN ở trên giữ nguyên. Hai điểm (2) và (3) khác seed 36, cần ghi nhận khi so sánh.

**RT-DETR chuyển sang DataParallel (15/11/2026), vẫn 2 GPU:** bản torchrun vẫn chậm. Theo `kaggle-train-rtdetr-peking.ipynb` (đã train nhanh trên Kaggle): 1 tiến trình duy nhất bọc model bằng `DataParallel`, mỗi batch 16 ảnh (R50) / 8 ảnh (R101) chia đều cho 2 GPU, 4 worker nạp ảnh dùng trọn 4 vCPU (torchrun chạy 2 tiến trình train + 2×2 worker tranh nhau 4 vCPU). Ảnh được tiền xử lý bằng bản native (resize 640 bilinear, /255): ra đúng cùng tensor với `RTDetrImageProcessor` nhưng nhanh ~3 lần (6,8 so với 21,7 ms/ảnh). Log có thêm `chờ data`. Siêu tham số, effective batch 16, các sửa NaN giữ nguyên. Đổi `RT_MULTI_GPU = 'ddp'` để quay về torchrun; checkpoint dùng được qua lại giữa hai chế độ.

Chỉ khác: `seed`, và `save_period=-1` (không lưu `epochN.pt`) để tiết kiệm dung lượng 20 GB của `/kaggle/working` — không ảnh hưởng kết quả train.

### Cách tận dụng T4 ×2
- **YOLO:** mỗi run 1 GPU, 2 GPU chạy 2 run song song.
- **RT-DETR:** 1 run dùng cả 2 GPU cùng lúc, nên chỉ bắt đầu khi cả 2 GPU cùng rảnh.
- `QUEUE_ORDER = 'yolo_first'` (mặc định): chạy hết YOLO theo cặp rồi tới RT-DETR. `'rtdetr_first'`: ngược lại. `'as_listed'`: theo thứ tự trong `RUNS` (một GPU có thể đứng chờ nếu RT-DETR nằm xen giữa các run YOLO).

### Thiết lập Kaggle (thanh bên phải)
1. **Accelerator:** GPU T4 ×2. **Internet:** ON (tải ultralytics, weights YOLO, PekingU từ Hugging Face).
2. **Input:** thêm dataset `raw_combined` và `curated_combined` (định dạng YOLO: `images/{train,val}`, `labels/{train,val}`).
3. **Resume qua nhiều phiên:** sau mỗi phiên, thêm *Output* của version trước làm Input — notebook tự tìm `projects/finding_seed/seed85/` và chép về để train tiếp.
4. Bấm **Save Version → Save & Run All (Commit)**. Notebook tự dừng an toàn trước giới hạn 12 h (mặc định 11.3 h), mỗi run dừng ngay sau khi lưu xong `last.pt` của một epoch.

### Ước lượng thời gian
| Run | Giờ / run |
|---|---|
| yolo11n, yolo26n (1 T4) | ~3–4.5 h (ước lượng thô) |
| yolo11m, yolo26m (1 T4) | ~10–11 h (đo được ~11–13 phút/epoch) |
| RT-DETR-R50 (1 T4, batch 2) | đo được ~45–60 phút/epoch → ~40–50 h |
| RT-DETR-R101 (1 T4, batch 1) | đo được ~74 phút/epoch → ~60 h |
| RT-DETR trên 2 T4 | chưa đo; kỳ vọng nhanh hơn khoảng 2 lần hoặc hơn nhờ 2 GPU + micro-batch lớn hơn — xem `epoch_seconds` trong `results.csv` |

RT-DETR là phần tốn thời gian nhất: mỗi run cần nhiều phiên Kaggle (tự resume giữa các phiên). Bật/tắt run ở cell cấu hình để chia theo phiên hoặc chia giữa các tài khoản.

## 0. Cấu hình — bật/tắt từng run

In [ ]:
# ===================== CẤU HÌNH — chỉ cần sửa cell này =====================
SEED = 85
EPOCHS = 50

# Bật/tắt từng run (True = chạy, False = bỏ qua). Thứ tự ở đây = thứ tự trong hàng đợi.
# Run đã xong (có done.json VÀ results.csv đủ EPOCHS dòng) luôn được bỏ qua, nên để True cũng không sao.
RUNS = {
    'raw_yolo11n':       True,
    'curated_yolo11n':   True,
    'raw_yolo26n':       True,
    'curated_yolo26n':   True,
    'raw_rtdetr50':      True,   # RT-DETR-R50vd
    'curated_rtdetr50':  True,
    'raw_yolo11m':       True,
    'curated_yolo11m':   True,
    'raw_yolo26m':       True,
    'curated_yolo26m':   True,
    'raw_rtdetr100':     True,   # RT-DETR-R101vd (giữ tên rtdetr100 cho khớp evaluator/report)
    'curated_rtdetr100': True,
}

GPUS = [0, 1]                 # YOLO: mỗi run 1 GPU (2 run song song). RT-DETR: 1 run dùng CẢ 2 GPU.
QUEUE_ORDER = 'yolo_first'    # 'yolo_first' | 'rtdetr_first' | 'as_listed' (theo thứ tự trong RUNS)
RT_MULTI_GPU = 'dp'           # RT-DETR trên 2 GPU: 'dp' = 1 tiến trình DataParallel (như kaggle-train-rtdetr-peking.ipynb,
                              #   đo được nhanh hơn trên Kaggle 4 vCPU) | 'ddp' = torchrun 2 tiến trình (bản cũ)
RT_EFFECTIVE_BATCH = 16       # RT-DETR: số GPU x batch/GPU x accum = 16 ảnh / bước optimizer (như seed 42)
SESSION_BUDGET_HOURS = 11.3   # dừng an toàn trước giới hạn 12h của Kaggle (tính từ lúc chạy cell này)
MIN_START_MINUTES = 20        # không khởi động run mới nếu còn ít hơn chừng này phút
HARD_KILL_GRACE_MINUTES = 15  # quá budget + grace mà vẫn chạy thì kill cứng
POLL_SECONDS = 300            # chu kỳ in tiến độ

RAW_DIR = '/kaggle/input/datasets/doogichi/transmission-line-defect-raw-dataset'
CURATED_DIR = '/kaggle/input/datasets/tydapchai/finding-nemo-human-reviewed-6cls/finding_nemo_human_reviewed_6cls'
# Như seed 42: chép dataset của run RT-DETR sang ổ NVMe local trước khi train (đọc /kaggle/input chậm).
# /tmp mất khi hết phiên, mỗi phiên chép lại (bỏ qua file đã có). Đặt None để đọc thẳng /kaggle/input.
RT_LOCAL_DATA = '/tmp/finding_seed_data'
# ===========================================================================

import time
from pathlib import Path

SESSION_START = time.time()
DEADLINE_TS = SESSION_START + SESSION_BUDGET_HOURS * 3600
# WORK là scratch/local staging, KHÔNG phải Camber Stash.
WORK = Path('/kaggle/working/projects/finding_seed')
STASH_ROOT = f'stash://bunpmc/projects/finding_seed/seed{SEED}'
SEED_DIR = WORK / f'seed{SEED}'
SCRIPTS = WORK / 'seed_scripts'
for folder in (SEED_DIR, SCRIPTS, WORK / 'weights', WORK / 'data', WORK / 'coco'):
    folder.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['insulator', 'insulator_broken', 'insulator_flashover',
               'bird_nest', 'foreign_object', 'broken_strand']
SPECS = {
    'yolo11n':   dict(kind='yolo', weights='yolo11n.pt', batch=16),
    'yolo11m':   dict(kind='yolo', weights='yolo11m.pt', batch=8),
    'yolo26n':   dict(kind='yolo', weights='yolo26n.pt', batch=16),
    'yolo26m':   dict(kind='yolo', weights='yolo26m.pt', batch=8),
    # RT-DETR: batch = micro-batch TRÊN MỖI GPU, bằng đúng micro-batch của seed 42 (R50 8, R101 4).
    # 2 GPU: R50 2 x 8 x accum 1 = 16, R101 2 x 4 x accum 2 = 16 ảnh / bước optimizer (= 8x2 / 4x4 của seed 42).
    # Nếu hết VRAM, hàng đợi tự chia đôi batch và chạy lại (lưu ở <run>/batch_override.json).
    'rtdetr50':  dict(kind='rtdetr', variant='r50vd', batch=8),
    'rtdetr100': dict(kind='rtdetr', variant='r101vd', batch=4),
}
unknown = [k for k in RUNS if k.split('_', 1)[1] not in SPECS or k.split('_', 1)[0] not in ('raw', 'curated')]
assert not unknown, f'Run key không hợp lệ: {unknown}'
assert QUEUE_ORDER in ('yolo_first', 'rtdetr_first', 'as_listed'), QUEUE_ORDER
ENABLED = [k for k, on in RUNS.items() if on]
print(f'Seed {SEED} | bật {len(ENABLED)}/12 run | GPU {GPUS} | dừng lúc {time.strftime("%H:%M", time.localtime(DEADLINE_TS))}')
print('\n'.join(f'  {"[x]" if on else "[ ]"} {k}' for k, on in RUNS.items()))

## 1. Cài thư viện và kiểm tra GPU

In [ ]:
import subprocess, sys
import torch

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('ultralytics==8.4.172')
NEED_RTDETR = any(SPECS[k.split('_', 1)[1]]['kind'] == 'rtdetr' for k in ENABLED)
if NEED_RTDETR:
    # Cùng dải phiên bản với kaggle_train_rtdetr_peking.ipynb (seed 42) để state_dict cùng tên khóa.
    pip('transformers>=4.44.0,<4.48.0', 'accelerate<1.0.0', 'pycocotools')

# ultralytics cần torchvision.ops.nms chạy được trên CUDA; tự sửa nếu bản torchvision lệch torch.
check = 'import torch, torchvision; b=torch.rand(4,4,device="cuda"); b[:,2:]+=1; torchvision.ops.nms(b, torch.rand(4,device="cuda"), 0.5); print(torchvision.__version__)'
res = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
if res.returncode != 0:
    major, minor = (int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    print('torchvision lỗi, cài lại bản khớp torch', torch.__version__)
    pip(f'torchvision==0.{minor + 15}.*')
    res = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
    assert res.returncode == 0, res.stderr[-2000:]
print('torch', torch.__version__, '| torchvision', res.stdout.strip())
assert torch.cuda.is_available(), 'Chưa bật GPU'
print(f'{torch.cuda.device_count()} GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert max(GPUS) < torch.cuda.device_count(), f'GPUS={GPUS} nhưng chỉ có {torch.cuda.device_count()} GPU'

## 2. Dò dataset raw / curated trong `/kaggle/input`

In [ ]:
import os, yaml

def find_dataset_dirs():
    found = []
    for root, dirs, _ in os.walk('/kaggle/input'):
        depth = Path(root).relative_to('/kaggle/input').parts
        if 'images' in dirs and 'labels' in dirs and (Path(root) / 'images' / 'train').is_dir():
            found.append(Path(root))
            dirs[:] = []
        elif len(depth) >= 5 or Path(root).name in ('images', 'labels'):
            dirs[:] = []
    return found

def pick(name, override, candidates):
    if override:
        return Path(override)
    matches = [p for p in candidates if name in str(p).lower()]
    if len(matches) != 1:
        raise RuntimeError(f'Không dò được dataset {name!r}: {matches or candidates}. Hãy đặt {name.upper()}_DIR ở cell cấu hình.')
    return matches[0]

needed = sorted({k.split('_', 1)[0] for k in ENABLED})
candidates = find_dataset_dirs()
DATASET_DIRS = {}
DATA_YAMLS = {}
for ds in needed:
    root = pick(ds, RAW_DIR if ds == 'raw' else CURATED_DIR, candidates)
    counts = {f'{kind}/{split}': len(list((root / kind / split).glob('*')))
              for kind in ('images', 'labels') for split in ('train', 'val')}
    assert all(counts.values()), f'{root} thiếu split: {counts}'
    DATASET_DIRS[ds] = root
    DATA_YAMLS[ds] = WORK / 'data' / f'{ds}.yaml'
    DATA_YAMLS[ds].write_text(yaml.safe_dump({
        'path': str(root), 'train': 'images/train', 'val': 'images/val',
        'names': dict(enumerate(CLASS_NAMES)),
    }, sort_keys=False), encoding='utf-8')
    print(f'{ds:8} {root}  {counts}')

## 3. Ưu tiên checkpoint trong local `/kaggle/working` và đường dẫn cũ

Sau đó cell 3b sẽ thử restore từ Camber Stash nếu local không có checkpoint.


In [ ]:
import csv, shutil

def epochs_done(run_dir):
    f = run_dir / 'results.csv'
    return max(0, len(f.read_text(encoding='utf-8').strip().splitlines()) - 1) if f.is_file() else 0

def is_done(run_dir):
    # done.json chỉ hợp lệ khi results.csv đủ EPOCHS dòng (trước đây có run bị ghi done.json giả).
    return (run_dir / 'done.json').is_file() and epochs_done(run_dir) >= EPOCHS

def invalidate_fake_done():
    for run_key in RUNS:
        d = SEED_DIR / run_key
        f = d / 'done.json'
        if f.is_file() and epochs_done(d) < EPOCHS:
            f.rename(d / 'done_invalid.json')
            print(f'  [DONE GIẢ] {run_key}: chỉ có {epochs_done(d)}/{EPOCHS} epoch -> đổi done.json thành done_invalid.json')

def has_checkpoint(run_dir):
    return (run_dir / 'weights' / 'last.pt').is_file()

# Chuyển checkpoint từ đường dẫn cũ sang projects/finding_seed/seed85 một lần.
# Tuyệt đối không ghi đè checkpoint đích đã có.
legacy_dir = Path('/kaggle/working/seed_runs') / f'seed{SEED}'
if legacy_dir.is_dir() and legacy_dir.resolve() != SEED_DIR.resolve():
    for old_run in legacy_dir.iterdir():
        if not old_run.is_dir():
            continue
        new_run = SEED_DIR / old_run.name
        if not new_run.exists():
            shutil.copytree(old_run, new_run)
            print(f'  MIGRATED {old_run.name}: {old_run} -> {new_run}')
        elif is_done(new_run) or has_checkpoint(new_run):
            print(f'  KEEP {old_run.name}: checkpoint đã có ở đường dẫn mới')
        else:
            # Chỉ bổ sung file còn thiếu, không overwrite file đích.
            for old_file in old_run.rglob('*'):
                if old_file.is_file():
                    new_file = new_run / old_file.relative_to(old_run)
                    if not new_file.exists():
                        new_file.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(old_file, new_file)
            print(f'  MERGED {old_run.name}: bổ sung file thiếu từ đường dẫn cũ')

# Ưu tiên Working; Input chỉ là phương án dự phòng nếu Working trống.
sources = []
input_root = Path('/kaggle/input')
if input_root.is_dir():
    for top in input_root.iterdir():
        for suffix in (f'seed_runs/seed{SEED}', f'projects/finding_seed/seed{SEED}'):
            sources.extend(p for p in top.rglob(suffix) if p.is_dir())
print('Thư mục lưu run:', SEED_DIR)
print('Nguồn backup trong Input:', [str(p) for p in sources] or 'không có (không sao nếu Working còn checkpoint)')

for run_key in RUNS:
    local = SEED_DIR / run_key
    if is_done(local) or has_checkpoint(local):
        print(f'  WORKING {run_key}: done={is_done(local)} epochs={epochs_done(local)} last.pt={has_checkpoint(local)}')
        continue
    candidates = [p / run_key for p in sources if (p / run_key).is_dir()]
    if candidates:
        best = max(candidates, key=lambda p: (is_done(p), epochs_done(p)))
        # Không xóa local và không ghi đè checkpoint có sẵn.
        shutil.copytree(best, local, dirs_exist_ok=True)
        print(f'  RESTORED {run_key}: done={is_done(local)} epochs={epochs_done(local)} <- {best}')
    else:
        print(f'  NEW {run_key}: chưa có checkpoint trong Working hoặc Input')

invalidate_fake_done()


## 3b. Camber Stash: tự cài CLI, xác thực qua Kaggle Secret, bỏ qua run chưa có trên Stash

Nếu checkpoint đã có trong `/kaggle/working`, notebook sẽ giữ nguyên. Khi Stash chưa có run, restore sẽ bỏ qua; các run được upload sau khi huấn luyện.


In [ ]:
# ===== CAMBER STASH: auto-install CLI + Kaggle Secrets authentication =====
# Kaggle -> Add-ons -> Secrets: CAMBER_API_KEY, bật quyền cho notebook.
# Không hardcode hay in API key vào notebook / log.
import os, shutil, subprocess
from pathlib import Path

if not os.environ.get('CAMBER_API_KEY'):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('CAMBER_API_KEY')
        if token:
            os.environ['CAMBER_API_KEY'] = token
    except Exception as exc:
        raise RuntimeError('Không đọc được Kaggle Secret CAMBER_API_KEY. Tạo Secret và cấp quyền notebook, hoặc đặt biến môi trường CAMBER_API_KEY.') from exc

if not os.environ.get('CAMBER_API_KEY'):
    raise RuntimeError('Thiếu CAMBER_API_KEY. Hãy tạo Kaggle Secret tên CAMBER_API_KEY rồi bật quyền notebook.')

# Camber CLI là chương trình riêng; có API key không tự sinh ra lệnh camber.
if not shutil.which('camber'):
    print('Đang cài Camber CLI từ installer chính thức của CamberCloud...')
    install = subprocess.run(
        ['bash', '-lc', 'curl -fsSL https://cli.cambercloud.com/install-v2.sh | bash'],
        capture_output=True, text=True)
    if install.returncode:
        raise RuntimeError('Cài Camber CLI thất bại: ' + (install.stderr or install.stdout)[-1500:])
    # Installer có thể thêm binary vào ~/.local/bin hoặc ~/bin trong phiên shell khác.
    for directory in [Path.home() / '.local/bin', Path.home() / 'bin', Path('/usr/local/bin')]:
        if directory.exists():
            os.environ['PATH'] = str(directory) + os.pathsep + os.environ.get('PATH', '')
    if not shutil.which('camber'):
        import re
        # Tìm binary trong HOME (không đụng tới checkpoint / GPU).
        candidates = list(Path.home().glob('**/camber'))
        candidates = [p for p in candidates if p.is_file() and os.access(p, os.X_OK)]
        if candidates:
            os.environ['PATH'] = str(candidates[0].parent) + os.pathsep + os.environ.get('PATH', '')
    if not shutil.which('camber'):
        raise RuntimeError('Installer chạy xong nhưng không tìm thấy binary camber trên PATH. Xem hướng dẫn: https://docs.cambercloud.com/docs/camber-cli/installation/')

print('Camber CLI:', shutil.which('camber'))
# API key được CLI tự lấy từ os.environ trong tiến trình con.
auth = subprocess.run(['camber', 'me'], capture_output=True, text=True)
if auth.returncode:
    raise RuntimeError('Camber authentication thất bại (không in token): ' + (auth.stderr or auth.stdout)[-1200:])
print('Camber authentication OK')

def camber_cmd(*parts, check=True):
    cmd = ['camber', 'stash', *map(str, parts)]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if check and p.returncode:
        raise RuntimeError(f"Camber Stash command failed: {' '.join(cmd)}\n{p.stderr or p.stdout}")
    return p

# Kiểm tra root quyền ghi và tạo các thư mục đích nếu thiếu.
base = 'stash://bunpmc'
parent_projects = base + '/projects'
parent = parent_projects + '/finding_seed'
for target in (parent_projects, parent, STASH_ROOT):
    if camber_cmd('test', '-d', target, check=False).returncode != 0:
        camber_cmd('mkdir', target)
camber_cmd('ls', STASH_ROOT)
print('Camber destination:', STASH_ROOT)

def restore_stash_run(run_key):
    dest = SEED_DIR / run_key
    # Tuyệt đối không ghi đè run đã hoàn thành / checkpoint đang có trong Working.
    if is_done(dest) or has_checkpoint(dest):
        print(f'[LOCAL KEPT] {run_key}: checkpoint/done.json đã có trong Working')
        return

    remote = f'{STASH_ROOT}/{run_key}'
    # Stash có thể chưa có run này (đặc biệt lần đầu upload seed85).
    # Không tin tuyệt đối vào `camber stash test -d`: thử copy và xử lý
    # riêng trường hợp source không tồn tại, KHÔNG nuốt lỗi permission/network.
    dest.mkdir(parents=True, exist_ok=True)
    result = camber_cmd('cp', '-r', remote + '/', str(dest) + '/', check=False)
    if result.returncode == 0:
        print(f'[STASH RESTORED] {run_key}: last.pt={has_checkpoint(dest)}; done={is_done(dest)}')
        return
    error = (result.stderr or result.stdout or '').strip()
    if 'source path does not exist' in error.lower() or 'source does not exist' in error.lower():
        print(f'[STASH SKIP] {run_key}: chưa có trên Camber Stash; giữ local và tiếp tục')
        return
    raise RuntimeError(f'Không thể restore {remote} từ Camber Stash: {error}')


def sync_stash_run(run_key):
    source = SEED_DIR / run_key
    if not source.is_dir():
        return
    remote = f'{STASH_ROOT}/{run_key}'
    # Tạo đích nếu chưa có; lệnh mkdir của Camber có thể báo lỗi khi đã tồn tại.
    probe = camber_cmd('ls', remote, check=False)
    if probe.returncode != 0:
        mk = camber_cmd('mkdir', remote, check=False)
        if mk.returncode != 0:
            recheck = camber_cmd('ls', remote, check=False)
            if recheck.returncode != 0:
                raise RuntimeError(f'Không tạo được Stash directory {remote}: {mk.stderr or mk.stdout}')
    camber_cmd('cp', '-r', str(source) + '/', remote + '/')
    print(f'[STASH SAVED] {run_key} -> {remote}')

for key in RUNS:
    restore_stash_run(key)
invalidate_fake_done()
print('Camber restore check complete.')


## 4. Script train YOLO (chạy trong tiến trình con, 1 GPU)

In [ ]:
script = r'''
import argparse, csv, json, os, sys, time
from pathlib import Path

p = argparse.ArgumentParser()
p.add_argument('--run-key', required=True)
p.add_argument('--data', required=True)
p.add_argument('--weights', required=True)
p.add_argument('--batch', type=int, required=True)
p.add_argument('--epochs', type=int, required=True)
p.add_argument('--seed', type=int, required=True)
p.add_argument('--project', required=True)
a = p.parse_args()

import torch
from ultralytics import YOLO, settings
try:
    settings.update({'raytune': False})  # tắt callback Ray Tune (log có cảnh báo get_trial_id)
except Exception:
    pass

DEADLINE = float(os.environ.get('DEADLINE_TS', 'inf'))
run_dir = Path(a.project) / a.run_key
last, best_pt, results = run_dir / 'weights' / 'last.pt', run_dir / 'weights' / 'best.pt', run_dir / 'results.csv'

def n_rows():
    if not results.is_file():
        return 0
    return max(0, len(results.read_text(encoding='utf-8').strip().splitlines()) - 1)

def write_done():
    rows = list(csv.DictReader(results.open(encoding='utf-8')))
    rows = [{k.strip(): v for k, v in r.items()} for r in rows]
    if len(rows) < a.epochs:
        print(f'[{a.run_key}] CHƯA XONG: results.csv mới có {len(rows)}/{a.epochs} epoch — KHÔNG ghi done.json', flush=True)
        sys.exit(3)
    best = max(rows, key=lambda r: float(r['metrics/mAP50(B)']))
    (run_dir / 'done.json').write_text(json.dumps({
        'run_key': a.run_key, 'seed': a.seed, 'epochs': len(rows), 'best_epoch': int(float(best['epoch'])),
        'best_val_mAP50': float(best['metrics/mAP50(B)']), 'best_val_mAP50_95': float(best['metrics/mAP50-95(B)']),
        'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, indent=2), encoding='utf-8')
    print(f'[{a.run_key}] COMPLETE ({len(rows)} epoch)', flush=True)

def retarget_checkpoint(path):
    # Ultralytics khi resume lấy project/name/save_dir/data từ chính checkpoint, không lấy từ tham số train().
    # Checkpoint tạo ở đường dẫn cũ (vd /kaggle/working/seed_runs/seed85) sẽ train tiếp và ghi kết quả
    # sang thư mục cũ, còn thư mục hiện tại đứng yên -> trông như "xong" mà không có epoch mới.
    # Ghi đè các đường dẫn này về thư mục hiện tại trước khi resume.
    ck = torch.load(path, map_location='cpu', weights_only=False)
    want = {'project': str(Path(a.project)), 'name': a.run_key, 'data': a.data,
            'save_dir': str(run_dir), 'exist_ok': True}
    changes = []
    def patch(args, where):
        if args is None:
            return
        d = args if isinstance(args, dict) else vars(args)
        for k, v in want.items():
            if k == 'save_dir' and k not in d:
                continue
            if str(d.get(k)) != str(v):
                changes.append(f'{where}.{k}: {d.get(k)} -> {v}')
                d[k] = v
    patch(ck.get('train_args'), 'train_args')
    for key in ('model', 'ema'):
        m = ck.get(key)
        if m is not None and getattr(m, 'args', None) is not None:
            patch(m.args, key)
    if changes:
        tmp = path.with_suffix('.tmp')
        torch.save(ck, tmp)
        os.replace(tmp, path)
        print(f'[{a.run_key}] sửa đường dẫn trong last.pt:\n  ' + '\n  '.join(changes), flush=True)
    return ck

done_rows = n_rows()
if done_rows >= a.epochs and best_pt.is_file():
    print(f'[{a.run_key}] đã đủ {done_rows} epoch; chỉ ghi done.json', flush=True)
    write_done()
    sys.exit(0)

resume = last.is_file()
if resume:
    ck = retarget_checkpoint(last)
    if ck.get('optimizer') is None:
        print(f'[{a.run_key}] LỖI: last.pt không còn optimizer nhưng mới có {done_rows}/{a.epochs} epoch — '
              f'không resume đúng nghĩa được. Cần train lại run này (xóa thư mục run) hoặc xử lý tay.', flush=True)
        sys.exit(4)
    ck_epoch = int(ck.get('epoch', -1)) + 1
    print(f'[{a.run_key}] RESUME từ epoch {ck_epoch + 1} (results.csv có {done_rows} dòng)', flush=True)
    del ck
else:
    print(f'[{a.run_key}] START seed={a.seed} batch={a.batch}', flush=True)

model = YOLO(str(last if resume else a.weights))
clock = {'t': time.time()}

def on_model_save(trainer):
    # Chạy ngay sau khi last.pt đã ghi xong. Thoát cứng ở đây để last.pt còn optimizer
    # (dừng mềm sẽ chạy final_eval -> strip_optimizer -> không resume được).
    if Path(trainer.save_dir).resolve() != run_dir.resolve():
        print(f'[{a.run_key}] LỖI: Ultralytics đang ghi vào {trainer.save_dir}, không phải {run_dir}', flush=True)
        os._exit(5)
    now = time.time()
    epoch_seconds, clock['t'] = now - clock['t'], now
    if trainer.epoch + 1 < trainer.epochs and now + 1.2 * epoch_seconds > DEADLINE:
        print(f'[PAUSE] {a.run_key}: saved epoch {trainer.epoch + 1}, next epoch would exceed the session budget', flush=True)
        os._exit(75)

model.add_callback('on_model_save', on_model_save)
model.train(
    data=a.data, epochs=a.epochs, imgsz=640, batch=a.batch, device=0,
    optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True, warmup_epochs=3.0,
    weight_decay=0.0005, close_mosaic=10, amp=True, seed=a.seed, deterministic=True,
    workers=2, project=a.project, name=a.run_key, exist_ok=True, save=True,
    save_period=-1, plots=True, resume=resume,
)
after = n_rows()
if after <= done_rows and done_rows < a.epochs:
    print(f'[{a.run_key}] LỖI: train() kết thúc nhưng results.csv không thêm epoch nào ({after}/{a.epochs})', flush=True)
    sys.exit(6)
write_done()
'''
(SCRIPTS / 'train_yolo.py').write_text(script.lstrip(), encoding='utf-8')
print('wrote', SCRIPTS / 'train_yolo.py')


## 5. Script train RT-DETR PekingU (mặc định DataParallel trên 2 GPU; vẫn hỗ trợ DDP hoặc 1 GPU)


In [ ]:
script = r'''
import argparse, csv, json, math, os, random, sys, time, traceback
from pathlib import Path

import numpy as np
import torch
import torch._utils
if not hasattr(torch._utils, '_chunk_or_narrow_cat'):
    def _chunk_or_narrow_cat(tensor, num_chunks, narrow_dim, cat_dim=0):
        return torch.cat(torch.chunk(tensor, num_chunks, dim=narrow_dim), dim=cat_dim)
    torch._utils._chunk_or_narrow_cat = _chunk_or_narrow_cat
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler

p = argparse.ArgumentParser()
p.add_argument('--run-key', required=True)
p.add_argument('--dataset-dir', required=True)
p.add_argument('--coco-dir', required=True)
p.add_argument('--variant', choices=['r50vd', 'r101vd'], required=True)
p.add_argument('--batch', type=int, required=True, help='micro-batch TRÊN MỖI GPU')
p.add_argument('--accum', type=int, required=True)
p.add_argument('--epochs', type=int, required=True)
p.add_argument('--seed', type=int, required=True)
p.add_argument('--project', required=True)
a = p.parse_args()

# torchrun đặt RANK / LOCAL_RANK / WORLD_SIZE. Chạy thường (1 GPU) thì WORLD = 1.
RANK = int(os.environ.get('RANK', '0'))
LOCAL_RANK = int(os.environ.get('LOCAL_RANK', '0'))
WORLD = int(os.environ.get('WORLD_SIZE', '1'))
MULTI = WORLD > 1
IS_MAIN = RANK == 0
# Chế độ DataParallel (như kaggle-train-rtdetr-peking.ipynb): 1 tiến trình dùng mọi GPU nhìn thấy, mỗi GPU nhận
# --batch ảnh. Chỉ bật khi không chạy qua torchrun.
DP = (not MULTI and os.environ.get('RTDETR_DP') == '1'
      and torch.cuda.is_available() and torch.cuda.device_count() > 1)
N_DEV = torch.cuda.device_count() if DP else WORLD   # số GPU cùng train 1 model

CLASS_NAMES = ['insulator', 'insulator_broken', 'insulator_flashover',
               'bird_nest', 'foreign_object', 'broken_strand']
DEADLINE = float(os.environ.get('DEADLINE_TS', 'inf'))
USE_AMP = (os.environ.get('RTDETR_AMP', '1') == '1'   # RTDETR_AMP=0 -> fp32 hoàn toàn
           and (torch.cuda.is_available() or os.environ.get('RTDETR_FORCE_AMP') == '1'))
FREEZE_BN = os.environ.get('RTDETR_FREEZE_BN', '0') == '1'  # mặc định KHÔNG khóa BN; khóa BN làm R50 học kém
WARMUP_STEPS = int(os.environ.get('RTDETR_WARMUP', '2000'))  # warmup tuyến tính theo bước optimizer, như RT-DETR gốc
MAX_BAD_RATIO = 0.05      # dừng nếu >5% micro-batch của một epoch vẫn NaN sau khi đã thử lại fp32
MAX_BAD_OPT_STEPS = 10    # dừng nếu 10 bước optimizer liên tiếp không dùng được
OOM_EXIT, PAUSE_EXIT = 42, 75
repo_id = f'PekingU/rtdetr_{a.variant}'
run_dir = Path(a.project) / a.run_key
wdir = run_dir / 'weights'
wdir.mkdir(parents=True, exist_ok=True)
last_path, best_path, csv_path = wdir / 'last.pt', wdir / 'best.pt', run_dir / 'results.csv'
status_path = run_dir / 'exit_status.json'
device = torch.device(f'cuda:{LOCAL_RANK}' if torch.cuda.is_available() else 'cpu')


def log(msg):
    if IS_MAIN:
        print(msg, flush=True)


def write_status(status, **extra):
    # Notebook đọc file này để biết run dừng vì lý do gì (mã thoát của torchrun không đáng tin).
    tmp = status_path.with_suffix(f'.tmp{RANK}')
    tmp.write_text(json.dumps({'status': status, 'rank': RANK, 'time': time.strftime('%Y-%m-%d %H:%M:%S'), **extra}),
                   encoding='utf-8')
    os.replace(tmp, status_path)


def is_oom(exc):
    oom_types = tuple(t for t in (getattr(torch, 'OutOfMemoryError', None),
                                  getattr(torch.cuda, 'OutOfMemoryError', None)) if t is not None)
    return isinstance(exc, oom_types) or 'out of memory' in str(exc).lower()


def main():
    if MULTI:
        import torch.distributed as dist
        dist.init_process_group(backend='nccl' if torch.cuda.is_available() else 'gloo')
    if torch.cuda.is_available():
        torch.cuda.set_device(device)

    # Cùng seed trên mọi GPU trước khi dựng model -> head mới khởi tạo giống hệt nhau trên các GPU.
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)
    # Khớp recipe seed 42: ưu tiên kernel cuDNN nhanh trên T4; seed vẫn cố định nhưng CUDA không bit-exact.
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

    from transformers import RTDetrConfig, RTDetrForObjectDetection

    class NativeProcessor:
        # Tương đương RTDetrImageProcessor của PekingU: resize 640x640 bilinear, /255, không normalize.
        def __call__(self, images, return_tensors='pt'):
            arr = np.asarray(images.convert('RGB').resize((640, 640), Image.BILINEAR), dtype=np.float32)
            return {'pixel_values': torch.from_numpy(arr.transpose(2, 0, 1) / 255.0).unsqueeze(0)}

    # RTDetrImageProcessor của HF chậm ~3 lần (đo 21,7 vs 6,8 ms/ảnh) mà ra đúng cùng tensor -> dùng bản native.
    # Kiểm tra trên 1 ảnh train: lệch thì quay về processor của HF.
    processor = NativeProcessor()
    try:
        from transformers import RTDetrImageProcessor
        hf_processor = RTDetrImageProcessor.from_pretrained(repo_id)
        first = json.loads((Path(a.coco_dir) / 'train.json').read_text(encoding='utf-8'))['images'][0]['file_name']
        with Image.open(Path(a.dataset_dir) / 'images' / 'train' / first) as im:
            im = im.convert('RGB')
            diff = float((hf_processor(images=im, return_tensors='pt')['pixel_values']
                          - processor(images=im)['pixel_values']).abs().max())
        if diff > 1e-5:
            log(f'native processor lệch HF ({diff:.2e}) -> dùng RTDetrImageProcessor')
            processor = hf_processor
        else:
            log(f'native processor khớp RTDetrImageProcessor (lệch {diff:.1e})')
    except Exception as exc:
        log(f'không so được với RTDetrImageProcessor, dùng native processor: {exc}')

    class CocoYolo(Dataset):
        def __init__(self, img_dir, json_path):
            self.img_dir = Path(img_dir)
            coco = json.loads(Path(json_path).read_text(encoding='utf-8'))
            self.images = {im['id']: im for im in coco['images']}
            self.ids = list(self.images)
            self.anns = {i: [] for i in self.ids}
            for ann in coco['annotations']:
                self.anns[ann['image_id']].append(ann)
        def __len__(self):
            return len(self.ids)
        def __getitem__(self, idx):
            info = self.images[self.ids[idx]]
            image = Image.open(self.img_dir / info['file_name']).convert('RGB')
            w, h = info['width'], info['height']
            boxes, classes = [], []
            for ann in self.anns[info['id']]:
                x, y, bw, bh = ann['bbox']
                boxes.append([(x + bw / 2) / w, (y + bh / 2) / h, bw / w, bh / h])
                classes.append(ann['category_id'])
            target = {
                'class_labels': torch.tensor(classes, dtype=torch.long) if classes else torch.zeros((0,), dtype=torch.long),
                'boxes': torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            }
            return processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0), target

    class ShardedEpochSampler(Sampler):
        # Thứ tự shuffle cố định theo (seed, epoch) và giống nhau trên mọi GPU; GPU thứ r lấy phần tử r, r+W, r+2W...
        # Danh sách được nối vòng cho chia hết số GPU -> mọi GPU có đúng cùng số batch.
        def __init__(self, n):
            self.n, self.epoch = n, 0
            self.per_rank = math.ceil(n / WORLD)
        def set_epoch(self, epoch):
            self.epoch = epoch
        def __len__(self):
            return self.per_rank
        def __iter__(self):
            g = torch.Generator()
            g.manual_seed(a.seed * 1000 + self.epoch)
            perm = torch.randperm(self.n, generator=g).tolist()
            perm = (perm + perm[:WORLD])[:self.per_rank * WORLD]
            return iter(perm[RANK::WORLD])

    def collate(batch):
        return torch.stack([b[0] for b in batch]), [b[1] for b in batch]

    def worker_init(worker_id):
        s = torch.initial_seed() % 2**32
        np.random.seed(s); random.seed(s)

    ds_dir, coco_dir = Path(a.dataset_dir), Path(a.coco_dir)
    train_ds = CocoYolo(ds_dir / 'images' / 'train', coco_dir / 'train.json')
    val_ds = CocoYolo(ds_dir / 'images' / 'val', coco_dir / 'val.json')
    workers = max(1, (os.cpu_count() or 4) // WORLD)
    workers = min(workers, 4)
    loader_batch = a.batch * N_DEV if DP else a.batch   # DP: 1 batch lớn được chia đều cho các GPU
    train_sampler = ShardedEpochSampler(len(train_ds))
    # persistent_workers: không phải dựng lại worker (và nạp lại COCO json) ở mỗi epoch.
    train_loader = DataLoader(train_ds, batch_size=loader_batch, sampler=train_sampler, num_workers=workers,
                              collate_fn=collate, pin_memory=True, worker_init_fn=worker_init,
                              persistent_workers=True, prefetch_factor=4)
    val_loader = DataLoader(torch.utils.data.Subset(val_ds, list(range(len(val_ds)))[RANK::WORLD]), batch_size=loader_batch,
                            shuffle=False, num_workers=workers, collate_fn=collate, pin_memory=True,
                            worker_init_fn=worker_init, persistent_workers=True)

    config = RTDetrConfig.from_pretrained(repo_id)
    config.num_labels = len(CLASS_NAMES)
    config.id2label = dict(enumerate(CLASS_NAMES))
    config.label2id = {n: i for i, n in enumerate(CLASS_NAMES)}
    model = RTDetrForObjectDetection.from_pretrained(repo_id, config=config, ignore_mismatched_sizes=True).to(device)
    fwd = model   # module dùng để forward; optimizer/checkpoint luôn dùng `model` gốc
    if DP:
        # Bản sao trên GPU phụ không có tham số riêng -> property device của HF lỗi; lấy theo tham số hoặc GPU hiện tại.
        def _safe_device(self):
            try:
                return next(self.parameters()).device
            except StopIteration:
                return torch.device(f'cuda:{torch.cuda.current_device()}')
        RTDetrForObjectDetection.device = property(_safe_device)

        class RTDetrDataParallel(torch.nn.DataParallel):
            # Chia ảnh + List[Dict] nhãn theo cùng ranh giới, chỉ gom loss (như kaggle-train-rtdetr-peking.ipynb).
            def scatter(self, inputs, kwargs, device_ids):
                pixels = kwargs['pixel_values']
                labels = kwargs.get('labels')
                chunks = torch.nn.parallel.scatter(pixels, device_ids)
                out_kwargs, i = [], 0
                for px, dev in zip(chunks, device_ids):
                    lbl = None
                    if labels is not None:
                        lbl = [{k: v.to(f'cuda:{dev}') for k, v in t.items()} for t in labels[i:i + px.size(0)]]
                        i += px.size(0)
                    out_kwargs.append({'pixel_values': px, 'labels': lbl})
                return tuple(() for _ in chunks), tuple(out_kwargs)

            def forward(self, *inputs, **kwargs):
                inputs, kwargs = self.scatter(inputs, kwargs, self.device_ids)
                replicas = self.replicate(self.module, self.device_ids[:len(kwargs)])
                outputs = self.parallel_apply(replicas, inputs, kwargs)
                loss = torch.stack([o.loss.reshape(()).to(self.output_device) for o in outputs]).mean()
                return type('Out', (), {'loss': loss})()

        fwd = RTDetrDataParallel(model)
        log(f'DataParallel trên {N_DEV} GPU: {[torch.cuda.get_device_name(i) for i in range(N_DEV)]}')

    backbone = [p_ for n, p_ in model.named_parameters() if p_.requires_grad and 'backbone' in n]
    others = [p_ for n, p_ in model.named_parameters() if p_.requires_grad and 'backbone' not in n]
    optimizer = optim.AdamW([{'params': backbone, 'lr': 1e-4}, {'params': others, 'lr': 1e-3}], weight_decay=0.0005)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=a.epochs, eta_min=1e-5)
    params = backbone + others
    n_param = sum(p_.numel() for p_ in params)

    # Bộ scale loss cho AMP, cùng quy tắc với torch GradScaler (x2 sau 2000 bước tốt, /2 khi gradient inf/NaN).
    # Tự quản lý để quyết định "bỏ bước" được đưa ra trên gradient ĐÃ gộp -> mọi GPU quyết định giống nhau.
    amp_state = {'scale': 65536.0 if USE_AMP else 1.0, 'good': 0}

    def atomic_save(obj, path):
        tmp = path.with_suffix('.tmp')
        torch.save(obj, tmp)
        os.replace(tmp, path)

    start_epoch, best_val = 0, float('inf')
    if last_path.is_file():
        ck = torch.load(last_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model_state'])
        optimizer.load_state_dict(ck['optimizer_state'])
        scheduler.load_state_dict(ck['scheduler_state'])
        sc = ck.get('scaler_state') or {}
        if USE_AMP and sc.get('scale'):
            amp_state['scale'] = float(sc['scale'])
            amp_state['good'] = int(sc.get('_growth_tracker', 0))
        start_epoch, best_val = ck['epoch'] + 1, ck['best_val_loss']
        del ck
        log(f'[{a.run_key}] RESUME from epoch {start_epoch + 1} | {N_DEV} GPU{" (DataParallel)" if DP else ""} x batch {a.batch} x accum {a.accum} '
            f'= {N_DEV * a.batch * a.accum} ảnh / bước optimizer')
    else:
        log(f'[{a.run_key}] START seed={a.seed} | {N_DEV} GPU{" (DataParallel)" if DP else ""} x batch {a.batch} x accum {a.accum} '
            f'= {N_DEV * a.batch * a.accum} ảnh / bước optimizer')

    def freeze_bn_stats(m):
        # Giữ running_mean/var của BN (encoder) ở giá trị pretrained; gamma/beta vẫn được train.
        for mod in m.modules():
            if isinstance(mod, torch.nn.modules.batchnorm._BatchNorm):
                mod.eval()

    def to_device(targets):
        return [{k: v.to(device) for k, v in t.items()} for t in targets]

    def forward_loss(pixels, targets, amp):
        # Trả về loss (tensor) hoặc None nếu loss/box dự đoán không hữu hạn.
        try:
            with torch.amp.autocast(device.type, enabled=amp):
                loss = fwd(pixel_values=pixels, labels=targets).loss.mean()
        except ValueError:  # matcher gặp box dự đoán NaN
            return None
        return loss if bool(torch.isfinite(loss)) else None

    def snapshot_bn_buffers():
        # Forward train cập nhật BN trước khi ta biết loss có NaN hay không. Giữ snapshot nhỏ để AMP lỗi
        # không đầu độc running_mean/running_var rồi làm cả các batch sau tiếp tục NaN.
        return [(m, m.running_mean.detach().clone() if m.running_mean is not None else None,
                 m.running_var.detach().clone() if m.running_var is not None else None,
                 m.num_batches_tracked.detach().clone() if m.num_batches_tracked is not None else None)
                for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]

    def restore_bn_buffers(snapshot):
        with torch.no_grad():
            for m, mean, var, tracked in snapshot:
                if mean is not None: m.running_mean.copy_(mean)
                if var is not None: m.running_var.copy_(var)
                if tracked is not None: m.num_batches_tracked.copy_(tracked)

    def try_loss(pixels, targets):
        # Thử AMP trước; nếu NaN/inf thì khôi phục BN rồi chạy lại riêng batch đó ở fp32.
        bn_before = snapshot_bn_buffers() if (USE_AMP and model.training and not FREEZE_BN) else None
        loss = forward_loss(pixels, targets, USE_AMP)
        if loss is not None:
            return loss, False
        if bn_before is not None:
            restore_bn_buffers(bn_before)
        if USE_AMP:
            loss = forward_loss(pixels, targets, False)
        return loss, True

    # ---- Đồng bộ nhiều GPU: mỗi GPU giữ một bản model, gradient được CỘNG GỘP ở mỗi bước optimizer. ----
    # Vector gộp = [gradient của mọi tham số | loss_sum, n_good, n_bad, n_fp32].
    N_STAT = 4
    # DataParallel đã tự cộng gradient về GPU chính; chỉ torchrun mới cần buffer all-reduce thủ công.
    gbuf = torch.zeros(n_param + N_STAT, device=device, dtype=torch.float32) if MULTI else None

    def gather_and_reduce(stats):
        offset = 0
        for p_ in params:
            n = p_.numel()
            if p_.grad is None:
                gbuf[offset:offset + n].zero_()
            else:
                gbuf[offset:offset + n].copy_(p_.grad.reshape(-1))
            offset += n
        gbuf[n_param:] = torch.tensor(stats, device=device, dtype=torch.float32)
        if MULTI:
            dist.all_reduce(gbuf, op=dist.ReduceOp.SUM)
        return gbuf[n_param:].tolist()

    def scatter_grads(divisor):
        offset = 0
        for p_ in params:
            n = p_.numel()
            g = gbuf[offset:offset + n].view_as(p_) / divisor
            if p_.grad is None:
                p_.grad = g.clone()
            else:
                p_.grad.copy_(g)
            offset += n

    def local_grads_finite():
        return all(p_.grad is None or bool(torch.isfinite(p_.grad).all()) for p_ in params)

    def unscale_local_grads(divisor):
        for p_ in params:
            if p_.grad is not None:
                p_.grad.div_(divisor)

    def reduce_values(values):
        t = torch.tensor(values, device=device, dtype=torch.float64)
        if MULTI:
            dist.all_reduce(t, op=dist.ReduceOp.SUM)
        return t.tolist()

    def main_says(flag):
        # Quyết định của GPU 0 được phát cho mọi GPU (để cùng dừng/cùng chạy tiếp).
        t = torch.tensor([1 if flag else 0], device=device, dtype=torch.int32)
        if MULTI:
            dist.broadcast(t, src=0)
        return bool(t.item())

    def sync_from_main():
        # Chép trọng số + buffer của GPU 0 sang các GPU khác (phòng lệch số học rất nhỏ tích lũy theo thời gian).
        # Trả về độ lệch checksum giữa các GPU TRƯỚC khi chép (0 = các bản model giống hệt nhau).
        if not MULTI:
            return 0.0
        with torch.no_grad():
            c = torch.stack([p_.double().abs().sum() for p_ in params]).sum().reshape(1)
            hi, lo = c.clone(), c.clone()
            dist.all_reduce(hi, op=dist.ReduceOp.MAX)
            dist.all_reduce(lo, op=dist.ReduceOp.MIN)
            for t in list(model.parameters()) + list(model.buffers()):
                if t.numel():
                    dist.broadcast(t, src=0)
        return float((hi - lo).item() / max(abs(hi.item()), 1e-12))

    sync_from_main()  # bảo đảm mọi GPU bắt đầu từ đúng cùng một bộ trọng số
    for epoch in range(start_epoch, a.epochs):
        t0 = time.time()
        train_sampler.set_epoch(epoch)
        # RNG khác nhau giữa các GPU (nhiễu denoising), cố định theo (seed, epoch, GPU) kể cả sau resume.
        rs = a.seed * 100003 + epoch * 101 + RANK
        random.seed(rs); np.random.seed(rs % 2**32); torch.manual_seed(rs); torch.cuda.manual_seed_all(rs)
        model.train()
        if FREEZE_BN:
            freeze_bn_stats(model)
        optimizer.zero_grad(set_to_none=True)
        n_steps = len(train_loader)
        opt_per_epoch = math.ceil(n_steps / a.accum)
        opt_step = epoch * opt_per_epoch           # bước optimizer toàn cục (đúng cả khi resume)
        epoch_lrs = [g['lr'] for g in optimizer.param_groups]  # lr cosine của epoch này
        tot = {'loss': 0.0, 'good': 0.0, 'bad': 0.0, 'fp32': 0.0, 'skipped_opt': 0}
        local = [0.0, 0.0, 0.0, 0.0]               # loss_sum, n_good, n_bad, n_fp32 kể từ lần gộp trước
        consec_bad_opt = 0
        t_wait, t_mark = 0.0, time.time()          # thời gian chờ DataLoader (CPU) trong epoch
        for step, (pixels, targets) in enumerate(train_loader):
            t_wait += time.time() - t_mark
            pixels, targets = pixels.to(device, non_blocking=True), to_device(targets)
            loss, retried = try_loss(pixels, targets)
            if loss is None:
                local[2] += 1                      # batch này không đóng góp gradient
            else:
                (loss.float() * (amp_state['scale'] / a.accum)).backward()
                local[0] += float(loss.item()); local[1] += 1; local[3] += 1 if retried else 0
            del loss
            if (step + 1) % a.accum == 0 or step + 1 == n_steps:
                if MULTI:
                    s_loss, s_good, s_bad, s_fp32 = gather_and_reduce(local)
                    grads_ok = s_good > 0 and bool(torch.isfinite(gbuf[:n_param]).all())
                else:
                    s_loss, s_good, s_bad, s_fp32 = local
                    grads_ok = s_good > 0 and local_grads_finite()
                local = [0.0, 0.0, 0.0, 0.0]
                tot['loss'] += s_loss; tot['good'] += s_good; tot['bad'] += s_bad; tot['fp32'] += s_fp32
                if grads_ok:
                    if MULTI:
                        scatter_grads(amp_state['scale'] * WORLD)  # torchrun: bỏ scale + trung bình giữa rank
                    else:
                        unscale_local_grads(amp_state['scale'])    # DataParallel đã lấy mean giữa GPU
                    warm = min(1.0, (opt_step + 1) / WARMUP_STEPS) if WARMUP_STEPS > 0 else 1.0
                    for g, lr in zip(optimizer.param_groups, epoch_lrs):
                        g['lr'] = lr * warm
                    torch.nn.utils.clip_grad_norm_(params, max_norm=0.1)
                    optimizer.step()
                    consec_bad_opt = 0
                    if USE_AMP:
                        amp_state['good'] += 1
                        if amp_state['good'] >= 2000:
                            amp_state['scale'] *= 2.0; amp_state['good'] = 0
                else:
                    tot['skipped_opt'] += 1; consec_bad_opt += 1
                    if USE_AMP and s_good > 0:      # gradient fp16 tràn -> giảm scale như GradScaler
                        amp_state['scale'] = max(amp_state['scale'] * 0.5, 1.0); amp_state['good'] = 0
                    log(f'  [SKIP] epoch {epoch + 1} step {step + 1}: bỏ 1 bước optimizer '
                        f'(batch tốt {int(s_good)}, batch NaN {int(s_bad)}, scale {amp_state["scale"]:.0f})')
                optimizer.zero_grad(set_to_none=True)
                opt_step += 1
                seen = tot['good'] + tot['bad']
                if consec_bad_opt >= MAX_BAD_OPT_STEPS or (tot['bad'] > 50 and tot['bad'] > MAX_BAD_RATIO * max(seen, 1)):
                    if not all(bool(torch.isfinite(p_).all()) for p_ in params):
                        raise RuntimeError(f'Trọng số đã NaN ở epoch {epoch + 1} step {step + 1}')
                    raise RuntimeError(f'Quá nhiều batch NaN kể cả sau khi thử lại fp32 '
                                       f'({int(tot["bad"])}/{int(seen)}), model đã phân kỳ')
                done_opt = opt_step - epoch * opt_per_epoch
                if done_opt % max(1, 200 // a.accum) == 0:
                    log(f'  epoch {epoch + 1} step {step + 1}/{n_steps} loss {tot["loss"] / max(tot["good"], 1):.4f} '
                        f'lr_head {optimizer.param_groups[1]["lr"]:.2e} fp32_retry {int(tot["fp32"])} '
                        f'nan_skip {int(tot["bad"])} ({(time.time() - t0) / 60:.0f} min, '
                        f'chờ data {100 * t_wait / max(time.time() - t0, 1e-9):.0f}%)')
            t_mark = time.time()
        for g, lr in zip(optimizer.param_groups, epoch_lrs):  # trả lr về giá trị cosine trước khi scheduler.step()
            g['lr'] = lr
        scheduler.step()
        drift = sync_from_main()

        model.eval()
        v_sum, v_n, v_bad = 0.0, 0, 0
        with torch.no_grad():
            for pixels, targets in val_loader:
                pixels, targets = pixels.to(device, non_blocking=True), to_device(targets)
                loss, _ = try_loss(pixels, targets)
                if loss is None:
                    v_bad += 1
                else:
                    v_sum += float(loss.item()); v_n += 1
        v_sum, v_n, v_bad = reduce_values([v_sum, v_n, v_bad])
        if v_n == 0:
            raise RuntimeError(f'Toàn bộ batch val đều NaN ở epoch {epoch + 1}')
        train_loss = tot['loss'] / max(tot['good'], 1)
        val_loss = v_sum / v_n
        epoch_seconds = time.time() - t0
        improved = val_loss < best_val
        best_val = min(best_val, val_loss)
        if IS_MAIN:
            if improved:
                atomic_save({'model_state': model.state_dict(), 'epoch': epoch, 'val_loss': val_loss,
                             'variant': a.variant, 'seed': a.seed}, best_path)
            atomic_save({'epoch': epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                         'scheduler_state': scheduler.state_dict(),
                         'scaler_state': {'scale': amp_state['scale'], '_growth_tracker': amp_state['good']},
                         'best_val_loss': best_val, 'world_size': WORLD, 'batch': a.batch, 'accum': a.accum}, last_path)
            new_file = not csv_path.is_file()
            with csv_path.open('a', newline='', encoding='utf-8') as f:
                w = csv.writer(f)
                if new_file:
                    w.writerow(['epoch', 'train_loss', 'val_loss', 'lr_backbone', 'lr_head', 'epoch_seconds'])
                w.writerow([epoch + 1, f'{train_loss:.5f}', f'{val_loss:.5f}', f'{optimizer.param_groups[0]["lr"]:.7f}',
                            f'{optimizer.param_groups[1]["lr"]:.7f}', f'{epoch_seconds:.0f}'])
            log(f'[{a.run_key}] epoch {epoch + 1}/{a.epochs} train {train_loss:.4f} val {val_loss:.4f}'
                f'{" *best*" if improved else ""} ({epoch_seconds / 60:.1f} min) | fp32_retry {int(tot["fp32"])} '
                f'nan_skip {int(tot["bad"])} opt_skip {tot["skipped_opt"]} val_nan {int(v_bad)} gpu_drift {drift:.1e} '
                f'| chờ data {t_wait / 60:.1f} min')
        pause = main_says(epoch + 1 < a.epochs and time.time() + 1.2 * epoch_seconds > DEADLINE)
        if pause:
            if IS_MAIN:
                log(f'[PAUSE] {a.run_key}: saved epoch {epoch + 1}, next epoch would exceed the session budget')
                write_status('paused', epoch=epoch + 1)
            return PAUSE_EXIT if not MULTI else 0

    if IS_MAIN:
        best = torch.load(best_path, map_location='cpu', weights_only=True)
        (run_dir / 'done.json').write_text(json.dumps({
            'run_key': a.run_key, 'seed': a.seed, 'epochs': a.epochs, 'best_epoch': best['epoch'] + 1,
            'best_val_loss': best['val_loss'], 'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, indent=2), encoding='utf-8')
        write_status('completed')
        log(f'[{a.run_key}] COMPLETE')
    return 0


if __name__ == '__main__':
    try:
        code = main()
    except BaseException as exc:  # noqa: BLE001 - cần bắt cả lỗi CUDA để ghi trạng thái trước khi thoát
        if isinstance(exc, SystemExit):
            raise
        traceback.print_exc()
        sys.stdout.flush(); sys.stderr.flush()
        if is_oom(exc):
            print(f'[OOM] GPU {RANK}: hết VRAM với batch {a.batch}/GPU', flush=True)
            write_status('oom', batch=a.batch)
            os._exit(OOM_EXIT)
        if not status_path.is_file():
            write_status('failed', error=f'{type(exc).__name__}: {exc}'[:500])
        os._exit(1)
    sys.stdout.flush()
    if MULTI:
        try:
            torch.distributed.destroy_process_group()
        except Exception:
            pass
    sys.exit(code)
'''
(SCRIPTS / 'train_rtdetr.py').write_text(script.lstrip(), encoding='utf-8')
print('wrote', SCRIPTS / 'train_rtdetr.py')


## 6. Tải pretrained weights và tạo COCO JSON cho RT-DETR

In [ ]:
import json
from PIL import Image

pending = [k for k in ENABLED if not is_done(SEED_DIR / k)]
print('Cần chạy:', pending or 'không còn run nào')

# 1) Chỉ tải pretrained weights nếu run chưa có checkpoint resume.
# Với last.pt trong Working, train_yolo.py sẽ tự YOLO(last.pt) + resume=True.
from ultralytics import YOLO
os.chdir(WORK / 'weights')
for model_key in sorted({k.split('_', 1)[1] for k in pending}):
    spec = SPECS[model_key]
    if spec['kind'] == 'yolo':
        spec['weights_path'] = str(WORK / 'weights' / spec['weights'])
        related = [k for k in pending if k.split('_', 1)[1] == model_key]
        needs_pretrained = any(not (SEED_DIR / k / 'weights' / 'last.pt').is_file() for k in related)
        if needs_pretrained:
            YOLO(spec['weights'])
        else:
            print(f'  {model_key}: dùng last.pt trong Working, bỏ qua tải pretrained')
    else:
        from huggingface_hub import snapshot_download
        snapshot_download(f"PekingU/rtdetr_{spec['variant']}")
os.chdir(WORK)

# 2) YOLO -> COCO JSON cho RT-DETR (giống kaggle_train_rtdetr_peking.ipynb: jpg/jpeg/png, bbox pixel xywh).
def yolo_to_coco(ds, split):
    out = WORK / 'coco' / ds / f'{split}.json'
    if out.is_file():
        return
    out.parent.mkdir(parents=True, exist_ok=True)
    root = DATASET_DIRS[ds]
    files = sorted(p for p in (root / 'images' / split).iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    images, anns = [], []
    for img_id, img in enumerate(files):
        with Image.open(img) as im:
            w, h = im.size
        images.append({'id': img_id, 'file_name': img.name, 'width': w, 'height': h})
        lbl = root / 'labels' / split / f'{img.stem}.txt'
        if lbl.is_file():
            for line in lbl.read_text(encoding='utf-8').splitlines():
                parts = line.split()
                if len(parts) >= 5:
                    xc, yc, bw, bh = (float(v) for v in parts[1:5])
                    anns.append({'id': len(anns), 'image_id': img_id, 'category_id': int(parts[0]),
                                 'bbox': [round((xc - bw / 2) * w, 2), round((yc - bh / 2) * h, 2), round(bw * w, 2), round(bh * h, 2)],
                                 'area': round(bw * w * bh * h, 2), 'iscrowd': 0})
    out.write_text(json.dumps({'images': images, 'annotations': anns,
                               'categories': [{'id': i, 'name': n} for i, n in enumerate(CLASS_NAMES)]}), encoding='utf-8')
    print(f'  {out}: {len(images):,} ảnh, {len(anns):,} box')

for ds in sorted({k.split('_', 1)[0] for k in pending if SPECS[k.split('_', 1)[1]]['kind'] == 'rtdetr'}):
    for split in ('train', 'val'):
        yolo_to_coco(ds, split)

# 3) Như seed 42 (Cell 2 của kaggle_train_rtdetr_peking.ipynb): chép images/labels của run RT-DETR sang NVMe.
from concurrent.futures import ThreadPoolExecutor
RT_DATASET_DIRS = {}
def copy_to_local(ds):
    src, dst = DATASET_DIRS[ds], Path(RT_LOCAL_DATA) / ds
    jobs = []
    for kind in ('images', 'labels'):
        for split in ('train', 'val'):
            for f in (src / kind / split).iterdir():
                out = dst / kind / split / f.name
                if f.is_file() and not (out.is_file() and out.stat().st_size == f.stat().st_size):
                    jobs.append((f, out))
    for kind in ('images', 'labels'):
        for split in ('train', 'val'):
            (dst / kind / split).mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(lambda j: shutil.copyfile(*j), jobs))
    n = {f'{k}/{s}': len(list((dst / k / s).iterdir())) for k in ('images', 'labels') for s in ('train', 'val')}
    m = {f'{k}/{s}': len(list((src / k / s).iterdir())) for k in ('images', 'labels') for s in ('train', 'val')}
    assert n == m, f'Chép thiếu file {ds}: {n} vs {m}'
    print(f'  {ds}: chép {len(jobs):,} file sang {dst} ({time.time() - t0:.0f}s) {n}')
    return dst

if RT_LOCAL_DATA:
    print('Free /tmp:', f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GB")
    for ds in sorted({k.split('_', 1)[0] for k in pending if SPECS[k.split('_', 1)[1]]['kind'] == 'rtdetr'}):
        RT_DATASET_DIRS[ds] = copy_to_local(ds)

## 7. Chạy hàng đợi: YOLO 1 GPU/run, RT-DETR 2 GPU/run

Log từng run: `projects/finding_seed/seed85/<run_key>/train.log`. Cell này chạy đến khi hết hàng đợi hoặc hết budget phiên.


In [ ]:
import signal, socket

N_GPU = len(GPUS)
SCHED_SLEEP_SECONDS = globals().get('SCHED_SLEEP_SECONDS', 10)   # chu kỳ kiểm tra tiến trình (giây)

def kind_of(run_key):
    return SPECS[run_key.split('_', 1)[1]]['kind']

def gpus_needed(run_key):
    # YOLO: 1 GPU / run. RT-DETR: 1 run chiếm toàn bộ GPU (train 1 model trên 2 GPU cùng lúc).
    return N_GPU if kind_of(run_key) == 'rtdetr' else 1

def rt_batch(run_key):
    # micro-batch TRÊN MỖI GPU. Nếu run từng hết VRAM thì dùng giá trị đã hạ trong batch_override.json.
    spec = SPECS[run_key.split('_', 1)[1]]
    f = SEED_DIR / run_key / 'batch_override.json'
    batch = json.loads(f.read_text())['batch'] if f.is_file() else spec['batch']
    accum = max(1, round(RT_EFFECTIVE_BATCH / (batch * N_GPU)))   # giữ effective batch = 16 như seed 42
    return batch, accum

def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

def command(run_key, gpus):
    ds, model_key = run_key.split('_', 1)
    spec = SPECS[model_key]
    if spec['kind'] == 'yolo':
        return [sys.executable, str(SCRIPTS / 'train_yolo.py'), '--run-key', run_key, '--data', str(DATA_YAMLS[ds]),
                '--weights', spec['weights_path'], '--batch', str(spec['batch']), '--epochs', str(EPOCHS),
                '--seed', str(SEED), '--project', str(SEED_DIR)]
    batch, accum = rt_batch(run_key)
    args = [str(SCRIPTS / 'train_rtdetr.py'), '--run-key', run_key,
            '--dataset-dir', str(globals().get('RT_DATASET_DIRS', {}).get(ds, DATASET_DIRS[ds])),
            '--coco-dir', str(WORK / 'coco' / ds), '--variant', spec['variant'], '--batch', str(batch),
            '--accum', str(accum), '--epochs', str(EPOCHS), '--seed', str(SEED), '--project', str(SEED_DIR)]
    if len(gpus) > 1 and globals().get('RT_MULTI_GPU', 'dp') == 'dp':   # 1 tiến trình DataParallel, thấy cả 2 GPU
        return [sys.executable, *args]
    if len(gpus) > 1:   # torchrun: mỗi GPU một tiến trình, cùng train MỘT model
        return [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node', str(len(gpus)),
                '--master_port', str(free_port()), *args]
    return [sys.executable, *args]

# Thứ tự hàng đợi. YOLO chạy từng cặp trên 2 GPU; RT-DETR cần cả 2 GPU nên chỉ bắt đầu khi cả hai cùng rảnh.
if QUEUE_ORDER == 'yolo_first':
    waiting = sorted(pending, key=lambda k: kind_of(k) == 'rtdetr')
elif QUEUE_ORDER == 'rtdetr_first':
    waiting = sorted(pending, key=lambda k: kind_of(k) != 'rtdetr')
else:
    waiting = list(pending)
print('Thứ tự chạy:', waiting or 'không còn run nào', flush=True)

free_gpus, active, outcomes = list(GPUS), {}, {}

def start(run_key):
    gpus = [free_gpus.pop(0) for _ in range(gpus_needed(run_key))]
    run_dir = SEED_DIR / run_key
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'exit_status.json').unlink(missing_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=','.join(map(str, gpus)), DEADLINE_TS=str(DEADLINE_TS),
               PYTHONUNBUFFERED='1', RTDETR_DP='1' if globals().get('RT_MULTI_GPU', 'dp') == 'dp' else '0')
    log = (run_dir / 'train.log').open('a', encoding='utf-8')
    log.write(f'\n===== session {time.strftime("%Y-%m-%d %H:%M:%S")} on GPU {gpus} =====\n')
    log.flush()
    proc = subprocess.Popen(command(run_key, gpus), stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(WORK),
                            start_new_session=True)   # nhóm tiến trình riêng để kill được cả các tiến trình con của torchrun
    active[run_key] = dict(proc=proc, gpus=gpus, t0=time.time(), log=log)
    extra = ''
    if kind_of(run_key) == 'rtdetr':
        batch, accum = rt_batch(run_key)
        extra = f' | {len(gpus)} GPU x batch {batch} x accum {accum}'
    print(f'[START] {run_key} trên GPU {gpus}{extra}', flush=True)

def finish(run_key, rc):
    info = active.pop(run_key)
    info['log'].close()
    free_gpus.extend(info['gpus']); free_gpus.sort()
    run_dir = SEED_DIR / run_key
    status = {}
    status_file = run_dir / 'exit_status.json'
    if status_file.is_file():
        try:
            status = json.loads(status_file.read_text(encoding='utf-8'))
        except Exception:
            status = {}
    retry = False
    if rc == 0 and is_done(run_dir):
        outcome = 'completed'
    elif rc == 75 or status.get('status') == 'paused':
        outcome = 'paused'
    elif status.get('status') == 'oom':
        batch, _ = rt_batch(run_key)
        if batch > 1:
            (run_dir / 'batch_override.json').write_text(json.dumps({'batch': batch // 2}), encoding='utf-8')
            outcome, retry = f'hết VRAM ở batch {batch}/GPU -> chạy lại với batch {batch // 2}/GPU', True
        else:
            outcome = 'failed (hết VRAM ngay cả với batch 1/GPU)'
    else:
        outcome = f'failed (exit {rc})'
    outcomes[run_key] = outcome
    print(f'[GPU{info["gpus"]}] {run_key}: {outcome} | epochs={epochs_done(run_dir)}/{EPOCHS}', flush=True)
    if retry:
        waiting.insert(0, run_key)
        return
    # Chỉ sync sau khi tiến trình đã dừng, tránh copy checkpoint lúc đang ghi.
    try:
        sync_stash_run(run_key)
    except Exception as e:
        print(f'[STASH ERROR] {run_key}: {e}', flush=True)
        outcomes[run_key] += ' / STASH SYNC FAILED'
    if outcome.startswith('failed'):
        print('   ' + '\n   '.join((run_dir / 'train.log').read_text(encoding='utf-8', errors='replace').splitlines()[-25:]), flush=True)

last_print = 0.0
while waiting or active:
    for run_key in list(active):
        rc = active[run_key]['proc'].poll()
        if rc is not None:
            finish(run_key, rc)
    can_start = time.time() < DEADLINE_TS - MIN_START_MINUTES * 60
    # Giữ đúng thứ tự: run đứng đầu chưa đủ GPU thì chờ, không cho run phía sau chen lên (tránh RT-DETR bị chờ mãi).
    while can_start and waiting and gpus_needed(waiting[0]) <= len(free_gpus):
        start(waiting.pop(0))
    if not active and (not can_start or not waiting):
        break
    left = (DEADLINE_TS - time.time()) / 60
    if left < -HARD_KILL_GRACE_MINUTES:
        for run_key, info in active.items():
            print(f'[KILL] {run_key} vượt budget quá {HARD_KILL_GRACE_MINUTES} phút', flush=True)
            try:
                os.killpg(info['proc'].pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
    if time.time() - last_print >= POLL_SECONDS / 2:
        last_print = time.time()
        running = [f'GPU{info["gpus"]}:{rk} ep {epochs_done(SEED_DIR / rk)}/{EPOCHS} ({(time.time() - info["t0"]) / 3600:.1f}h)'
                   for rk, info in active.items()]
        tail = f' | chờ: {waiting}' if waiting else ''
        print(f'{time.strftime("%H:%M")} | còn {left:.0f} phút | ' + (' | '.join(running) or 'idle') + tail, flush=True)
    time.sleep(SCHED_SLEEP_SECONDS)

if waiting:
    print('Chưa kịp chạy trong phiên này:', waiting, flush=True)
print('Kết quả phiên:', json.dumps(outcomes, indent=2, ensure_ascii=False))


## 8. Trạng thái 12 run (chạy lúc nào cũng được)

In [ ]:
import pandas as pd

def summarize(run_key):
    run_dir = SEED_DIR / run_key
    done = json.loads((run_dir / 'done.json').read_text()) if is_done(run_dir) else {}
    row = {'run': run_key, 'enabled': RUNS[run_key],
           'status': 'completed' if done else 'in progress' if epochs_done(run_dir) else 'pending',
           'epochs': f'{epochs_done(run_dir)}/{EPOCHS}'}
    f = run_dir / 'results.csv'
    if f.is_file():
        df = pd.read_csv(f)
        df.columns = [c.strip() for c in df.columns]
        if 'metrics/mAP50(B)' in df:
            i = df['metrics/mAP50(B)'].idxmax()
            row.update(best_epoch=int(df.loc[i, 'epoch']), val_mAP50=round(df.loc[i, 'metrics/mAP50(B)'], 4),
                       val_mAP50_95=round(df.loc[i, 'metrics/mAP50-95(B)'], 4))
        elif 'val_loss' in df:
            i = df['val_loss'].idxmin()
            row.update(best_epoch=int(df.loc[i, 'epoch']), val_loss=round(df.loc[i, 'val_loss'], 4))
    return row

status = pd.DataFrame([summarize(k) for k in RUNS])
status.to_csv(SEED_DIR / 'status.csv', index=False)
display(status)
camber_cmd('cp', str(SEED_DIR / 'status.csv'), STASH_ROOT + '/status.csv')
print('[STASH SAVED] status.csv')

## 9. Đóng gói best.pt để tải về

In [ ]:
import zipfile

# Gói nhẹ chỉ chứa best.pt + log để tải về đánh giá local (thư mục projects/finding_seed/seed85 đầy đủ vẫn nằm trong Working để resume).
zip_path = WORK / f'seed{SEED}_best_weights.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as z:
    for run_key in RUNS:
        run_dir = SEED_DIR / run_key
        for rel in ('weights/best.pt', 'results.csv', 'args.yaml', 'done.json'):
            if (run_dir / rel).is_file():
                z.write(run_dir / rel, f'seed{SEED}/{run_key}/{rel}')
    z.write(SEED_DIR / 'status.csv', f'seed{SEED}/status.csv')
print(f'{zip_path} ({zip_path.stat().st_size / 2**20:.0f} MB)')
camber_cmd('cp', str(zip_path), STASH_ROOT + '/' + zip_path.name)
print('[STASH SAVED] best weights ZIP')
print(f'Tổng dung lượng /kaggle/working: {sum(p.stat().st_size for p in WORK.rglob("*") if p.is_file()) / 2**30:.1f} GB (giới hạn 20 GB)')

## Sau khi chạy

- **Local scratch:** `/kaggle/working/projects/finding_seed/seed85/`.
- **Camber Stash:** `stash://bunpmc/projects/finding_seed/seed85/` lưu riêng từng run (`weights/last.pt`, `best.pt`, `results.csv`, `done.json`).
- Tự khôi phục từ Stash nếu local trống, không ghi đè checkpoint local hiện tại.
- Đồng bộ lên Stash sau mỗi run dừng/hoàn thành, `status.csv` và ZIP được đồng bộ ở các cell cuối.
- **Lưu ý:** nếu runtime bị crash giữa một run trước bước sync, có thể mất các epoch sau lần đồng bộ gần nhất; cần cấu hình Camber CLI và quyền truy cập từ môi trường đang chạy.
